In [ ]:
import os, asyncio, httpx
import pandas as pd
from binance import AsyncClient, BinanceSocketManager
from dotenv import load_dotenv


load_dotenv('../../.env')
BINANCE_KEY = os.getenv('BINANCE_KEY')
BINANCE_SECRET = os.getenv('BINANCE_SECRET')

client = await AsyncClient.create(BINANCE_KEY, BINANCE_SECRET)

In [2]:


# async def trade_history():
#     # bsm = BinanceSocketManager(client)

account_trades = await client.get_account()
# traded_symbols = {balance['asset'] + "USDT" for balance in account_trades['balances']}  # Adjust for different pairs
# traded_symbols = {bal['asset']: bal for bal in account_trades['balances'] if float(bal['free'])}
# traded_symbols = [bal for bal in account_trades['balances'] if float(bal['free'])]

# Fetch trades for each symbol
# tasks = [client.get_my_trades(symbol=symbol) for symbol in traded_symbols]
# all_trades = await asyncio.gather(*tasks, return_exceptions=True)
# ic(all_trades[0])

trade_history = [bal for bal in account_trades['balances'] if float(bal['free']) > 0 or float(bal['locked']) > 0]

df = pd.DataFrame(trade_history)  # noqa
df

NameError: name 'client' is not defined

In [190]:
df = df[(df['free'].astype(float) >= 1) | (df['locked'].astype(float) >= 1)]
# df['asset'].unique()

In [234]:
tasks = [client.get_my_trades(symbol=f'{symbol}USDT') for symbol in df['asset']]
trades = await asyncio.gather(*tasks, return_exceptions=True)
tradesdf = pd.concat([pd.DataFrame(trade) for trade in trades], ignore_index=True)
tradesdf['time'] = pd.to_datetime(tradesdf['time'], unit='ms')
tradesdf = tradesdf.set_index('id').sort_values(by='time').sort_values(by='id')
tradesdf

,symbol,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch
id,,,,,,,,,,,,
2710807,REDUSDT,36848173,-1,0.53980000,172.10000000,92.89958000,0.17210000,RED,2025-03-14 16:21:05.713,True,False,True
2710808,REDUSDT,36848173,-1,0.53980000,66.30000000,35.78874000,0.06630000,RED,2025-03-14 16:21:05.713,True,False,True
2710809,REDUSDT,36848173,-1,0.53980000,13.90000000,7.50322000,0.01390000,RED,2025-03-14 16:21:05.713,True,False,True
2710810,REDUSDT,36848173,-1,0.54010000,4.20000000,2.26842000,0.00420000,RED,2025-03-14 16:21:05.713,True,False,True
2710811,REDUSDT,36848173,-1,0.54010000,21.30000000,11.50413000,0.02130000,RED,2025-03-14 16:21:05.713,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
39390706,ZROUSDT,845588788,-1,3.22300000,103.50000000,333.58050000,0.10350000,ZRO,2025-03-27 06:59:30.675,True,False,True
39390707,ZROUSDT,845588788,-1,3.22300000,99.81000000,321.68763000,0.09981000,ZRO,2025-03-27 06:59:30.675,True,False,True
118591245,EGLDUSDT,1910149777,-1,18.85000000,9.22000000,173.79700000,0.00922000,EGLD,2025-03-27 12:32:15.535,True,False,True


In [237]:
# tradesdf['symbol'].unique()
tradesdf.columns

Index(['symbol', 'orderId', 'orderListId', 'price', 'qty', 'quoteQty',
       'commission', 'commissionAsset', 'time', 'isBuyer', 'isMaker',
       'isBestMatch'],
      dtype='object')

In [ ]:
tradesdf[tradesdf['isBuyer']].sample(10)
tradesdf[(tradesdf['isBuyer']) & (tradesdf['symbol'] == 'BANANAUSDT') & (tradesdf['orderId'] == 383583165)]

In [271]:
comm = 0.001
total = 19.41000000 * 11.87900000
# fee = total * comm
expense = total + 0.01187900
# fee
# total
expense

230.58326899999997

In [238]:
tradesdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 2710807 to 118591247
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   symbol           116 non-null    object        
 1   orderId          116 non-null    int64         
 2   orderListId      116 non-null    int64         
 3   price            116 non-null    object        
 4   qty              116 non-null    object        
 5   quoteQty         116 non-null    object        
 6   commission       116 non-null    object        
 7   commissionAsset  116 non-null    object        
 8   time             116 non-null    datetime64[ns]
 9   isBuyer          116 non-null    bool          
 10  isMaker          116 non-null    bool          
 11  isBestMatch      116 non-null    bool          
dtypes: bool(3), datetime64[ns](1), int64(2), object(6)
memory usage: 9.4+ KB


## Orders

In [31]:
basedf = pd.DataFrame(await client.get_all_orders(symbol='BANANAUSDT')).set_index('orderId')

In [50]:
ordersdf = basedf.copy()
ordersdf['time'] = pd.to_datetime(ordersdf['time'], unit='ms')
ordersdf['updateTime'] = pd.to_datetime(ordersdf['updateTime'], unit='ms')
ordersdf['workingTime'] = pd.to_datetime(ordersdf['workingTime'], unit='ms')
ordersdf = ordersdf.drop(columns=['workingTime', 'selfTradePreventionMode', 'isWorking'])
ordersdf[ordersdf['price'] == '0.00000000']
# ordersdf.loc[386446134]
# ordersdf

,symbol,orderListId,clientOrderId,price,origQty,executedQty,cummulativeQuoteQty,status,timeInForce,type,side,stopPrice,icebergQty,time,updateTime,origQuoteOrderQty
orderId,,,,,,,,,,,,,,,,
386446134,BANANAUSDT,-1,web_ae0fa6833e7542f59235a27afae844df,0.00000000,101.15800000,101.15800000,2104.11140000,FILLED,GTC,MARKET,SELL,0.00000000,0.00000000,2025-04-01 05:18:10.502,2025-04-01 05:18:10.502,0.00000000
386831406,BANANAUSDT,-1,web_af5aa18f2fdd45c9bd581b7f5b5c9be3,0.00000000,104.84600000,104.84600000,2102.46324000,FILLED,GTC,MARKET,BUY,0.00000000,0.00000000,2025-04-01 10:20:40.155,2025-04-01 10:20:40.155,2102.48038000


In [36]:
ordersdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8 entries, 376714213 to 387415847
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype          
---  ------                   --------------  -----          
 0   symbol                   8 non-null      object         
 1   orderListId              8 non-null      int64          
 2   clientOrderId            8 non-null      object         
 3   price                    8 non-null      object         
 4   origQty                  8 non-null      object         
 5   executedQty              8 non-null      object         
 6   cummulativeQuoteQty      8 non-null      object         
 7   status                   8 non-null      object         
 8   timeInForce              8 non-null      object         
 9   type                     8 non-null      object         
 10  side                     8 non-null      object         
 11  stopPrice                8 non-null      object         
 12  icebergQty     